# 03. Cobertura e Uso do Solo — MapBiomas + PRODES/INPE

Classificação da cobertura e uso do solo via **MapBiomas** e monitoramento do desmatamento via **PRODES** (INPE), cruzados com os focos de calor do notebook 01.

**Fonte primária — raster:** MapBiomas Coleção 7 (2021) — COG público no GCS (`mapbiomas-public`)  
**Fonte primária — estatísticas de área:** MapBiomas Coleção 8 (2023) — valores tabulares publicados em mapbiomas.org  
**Fonte secundária:** PRODES/INPE — série histórica oficial (Amazônia 2013–2023, Cerrado 2019–2023)  
**Método:** rasterio VSICURL sobre COG GCS → amostragem ponto-a-ponto nos focos de calor  
**Período:** focos INPE 2023 — mesmo período dos notebooks 01 e 02

> **Nota metodológica:** O raster de cobertura usado na amostragem é a Coleção 7 (ano-base 2021), a mais recente disponível publicamente no GCS. As estatísticas de área por bioma (seção 5) são da Coleção 8/2023 (valores tabulares). A Coleção 8 completa em raster não está no bucket público.

## 0. Dependências e configuração

> **Pré-requisito:** `pip install rasterio`  
> O raster MapBiomas Coleção 7 (2021) é lido diretamente do Google Cloud Storage via VSICURL (COG) — sem download completo.  
> Sem rasterio, o notebook usa imputação estatística por bioma com distribuições da Coleção 8/2023.

In [ ]:
import io, urllib3
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from pathlib import Path

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

try:
    import rasterio
    RASTERIO_OK = True
except ImportError:
    RASTERIO_OK = False
    print('[aviso] rasterio nao instalado — execute: pip install rasterio')
    print('        sem rasterio, serao usadas estatisticas tabulares MapBiomas')

MAPB_DIR   = Path('data/mapbiomas')
PRODES_DIR = Path('data/prodes')
MAPB_DIR.mkdir(parents=True, exist_ok=True)
PRODES_DIR.mkdir(parents=True, exist_ok=True)

ANO = 2023

# COG MapBiomas Colecao 7 / 2021 — mais recente disponivel no bucket publico GCS
# (Colecao 8 nao esta no bucket mapbiomas-public; estatisticas tabulares usadas na secao 5)
MAPB_COG_URL = (
    'https://storage.googleapis.com/mapbiomas-public/brasil/'
    'collection-7/lclu/coverage/brasil_coverage_2021.tif'
)
MAPB_COG_ANO = 2021   # ano de referencia do raster

MESES_PT = ['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez']

print('Configuracao OK')
print(f'MAPB_DIR   : {MAPB_DIR}')
print(f'PRODES_DIR : {PRODES_DIR}')
print(f'Periodo    : {ANO}  (raster cobertura: {MAPB_COG_ANO})')
print(f'Rasterio   : {RASTERIO_OK}')

## 1. Legenda e classes — MapBiomas Coleção 8

In [ ]:
# Legenda MapBiomas Colecao 8 (classes principais relevantes para incendio)
# Formato: codigo -> (grupo, descricao, cor_hex)
LEGENDA_MB = {
     1: ('Floresta Nativa',   'Floresta',                        '#1f4423'),
     3: ('Floresta Nativa',   'Formacao Florestal',              '#006400'),
     4: ('Floresta Nativa',   'Formacao Savânica',               '#00b050'),
     5: ('Floresta Nativa',   'Manguezal',                       '#687537'),
     6: ('Floresta Nativa',   'Floresta Alagada',                '#76a5af'),
     9: ('Silvicultura',      'Silvicultura',                    '#935132'),
    11: ('Veg. Natural',      'Campo Alagado e Area Pantanosa',  '#45c2a5'),
    12: ('Veg. Natural',      'Formacao Campestre',              '#b8af4f'),
    14: ('Veg. Natural',      'Pastagem Natural',                '#ffd966'),
    15: ('Pastagem',          'Pastagem',                        '#e974ed'),
    18: ('Agricultura',       'Agricultura',                     '#d5a6bd'),
    19: ('Agricultura',       'Lavoura Temporaria',              '#e787f8'),
    21: ('Agricultura',       'Mosaico de Usos',                 '#ffefc3'),
    22: ('Area nao Vegetada', 'Area nao Vegetada',               '#d4271e'),
    24: ('Area Urbana',       'Infraestrutura Urbana',           '#af2a2a'),
    25: ('Area nao Vegetada', 'Outra Area nao Vegetada',         '#d89f5c'),
    26: ('Corpo Dagua',       'Corpo Dagua',                     '#0000ff'),
    29: ('Area nao Vegetada', 'Afloramento Rochoso',             '#af2a2a'),
    30: ('Mineracao',         'Mineracao',                       '#9c0027'),
    33: ('Corpo Dagua',       'Rio, Lago e Oceano',              '#2532e4'),
    36: ('Agricultura',       'Lavoura Perene',                  '#d082de'),
    39: ('Agricultura',       'Soja',                            '#f5b3be'),
    40: ('Agricultura',       'Arroz',                           '#c71585'),
    41: ('Agricultura',       'Outras Lavouras Temporarias',     '#f54ca9'),
    49: ('Floresta Nativa',   'Restinga Arborea',                '#02d659'),
    50: ('Veg. Natural',      'Restinga Herbacea',               '#ad5100'),
}

# Grupos macros para visualizacao
GRUPOS_MB = {
    'Floresta Nativa':   ([1, 3, 4, 5, 6, 49],           '#006400'),
    'Veg. Natural':      ([11, 12, 14, 50],               '#b8af4f'),
    'Pastagem':          ([15],                           '#e974ed'),
    'Agricultura':       ([18, 19, 21, 36, 39, 40, 41],  '#d5a6bd'),
    'Silvicultura':      ([9],                            '#935132'),
    'Urbano/Mineracao':  ([22, 24, 25, 29, 30],          '#d4271e'),
    'Corpo Dagua':       ([26, 33],                       '#2532e4'),
}

# Mapeamento classe -> grupo
CLASS_TO_GRUPO = {}
for g, (cls_list, _) in GRUPOS_MB.items():
    for c in cls_list:
        CLASS_TO_GRUPO[c] = g

print(f'Classes na legenda    : {len(LEGENDA_MB)}')
print(f'Grupos simplificados  : {len(GRUPOS_MB)}')
print()
for grupo, (cls_list, cor) in GRUPOS_MB.items():
    nomes = [LEGENDA_MB[c][1] for c in cls_list if c in LEGENDA_MB]
    print(f'  {grupo:<22} classes={cls_list}  ({len(nomes)} subclasses)')

## 2. Leitura do raster MapBiomas via VSICURL

In [ ]:
# Abre o COG MapBiomas diretamente do GCS via VSICURL
# Colecao 7 / 2021 — unica versao disponivel no bucket publico mapbiomas-public

mapb_path = None

if RASTERIO_OK:
    vsicurl = f'/vsicurl/{MAPB_COG_URL}'
    print(f'Abrindo COG remoto: {MAPB_COG_URL}')
    try:
        with rasterio.open(vsicurl) as src:
            print(f'[ok] COG acessivel')
            print(f'     Colecao   : MapBiomas 7 / {MAPB_COG_ANO}')
            print(f'     Resolucao : {src.width} x {src.height} px')
            print(f'     CRS       : {src.crs}')
            print(f'     Dtype     : {src.dtypes[0]}')
            print(f'     Bounds    : {src.bounds}')
        mapb_path = vsicurl
    except Exception as e:
        print(f'[aviso] Acesso remoto falhou: {e}')
        print('        Usando imputacao estatistica por bioma')
else:
    print('[aviso] rasterio indisponivel — usando imputacao estatistica')

print(f'\nModo de dados: {"raster COG col-7/2021" if mapb_path else "imputacao estatistica col-8/2023"}')

## 3. Amostragem MapBiomas nos focos de calor (Notebook 01)

In [ ]:
# Carrega focos de calor consolidados pelo notebook 01
focos_csv = Path('data/inpe_queimadas/inpe_focos_2023_consolidado.csv')

if focos_csv.exists():
    df_focos = pd.read_csv(
        focos_csv,
        usecols=['latitude','longitude','bioma','frp','mes'],
        dtype={'frp': float, 'mes': int}
    )
    df_focos['bioma'] = df_focos['bioma'].str.strip()
    print(f'Focos carregados : {len(df_focos):,}  (de {focos_csv})')
    print(f'Biomas           : {sorted(df_focos["bioma"].unique())}')
    print(f'Periodo          : mes {df_focos["mes"].min()} a {df_focos["mes"].max()}')
else:
    print(f'[aviso] {focos_csv} nao encontrado — execute o notebook 01 primeiro')
    raise FileNotFoundError(
        f'{focos_csv}. Execute 01_focos_calor.ipynb para gerar o CSV consolidado.'
    )

df_focos[['latitude','longitude','bioma','frp','mes']].head()

In [ ]:
# Associa classe MapBiomas a cada foco de calor
# Estrategia A (rasterio OK): le o raster via overview ~1km para memoria e faz lookup vetorizado
# Estrategia B (fallback)   : imputacao estatistica pela distribuicao de area por bioma

df_focos['classe_mb'] = -1

if RASTERIO_OK and mapb_path:
    print(f'Amostrando {len(df_focos):,} pontos no raster MapBiomas {MAPB_COG_ANO}...')
    try:
        with rasterio.open(mapb_path) as src:
            # Le o raster a ~1km de resolucao (escala 1/32 do nativo ~30m)
            # O COG tem overviews [2,4,8,16,32,64,128,256,512] — rasterio escolhe o mais proximo
            OUT_W, OUT_H = 4850, 4950   # ~1km/pixel para o Brasil
            arr = src.read(
                1,
                out_shape=(1, OUT_H, OUT_W),
                resampling=rasterio.enums.Resampling.nearest
            ).squeeze()

            bounds = src.bounds
            nodata = src.nodata if src.nodata is not None else 0
            print(f'  Raster lido: {OUT_W}x{OUT_H} px (~1km/pixel)  shape={arr.shape}')

        # Lookup vetorizado: lon/lat -> col/row no array lido
        from rasterio.transform import from_bounds as tfrom_bounds
        t_arr = tfrom_bounds(bounds.left, bounds.bottom, bounds.right, bounds.top, OUT_W, OUT_H)

        lons = df_focos['longitude'].values
        lats = df_focos['latitude'].values
        cols = ((lons - t_arr.c) / t_arr.a).astype(int)
        rows = ((lats - t_arr.f) / t_arr.e).astype(int)
        cols = np.clip(cols, 0, OUT_W - 1)
        rows = np.clip(rows, 0, OUT_H - 1)

        classes = arr[rows, cols].astype(int)
        classes[classes == int(nodata)] = -1
        df_focos['classe_mb'] = classes

        n_val = (df_focos['classe_mb'] > 0).sum()
        print(f'  [ok] Classes validas: {n_val:,} ({100*n_val/len(df_focos):.1f}%)')
        print(f'  Classes encontradas: {sorted(set(classes[classes>0].tolist()))}')

    except Exception as e:
        print(f'  [aviso] Amostragem raster falhou: {e}')
        print('          Usando imputacao estatistica')

# Fallback: imputacao estatistica por bioma (distribuicoes MapBiomas Colecao 8 / 2023)
DISTRIB_BIOMA_MB = {
    'Amazônia':       {3: 0.835, 4: 0.009, 11: 0.008, 15: 0.089, 19: 0.020, 30: 0.010, 26: 0.029},
    'Cerrado':        {4: 0.521, 12: 0.075, 11: 0.018, 15: 0.208, 19: 0.092, 21: 0.048, 39: 0.038},
    'Caatinga':       {4: 0.452, 12: 0.109, 14: 0.059, 15: 0.224, 19: 0.095, 21: 0.061},
    'Mata Atlântica': {3: 0.280, 4: 0.052, 12: 0.028, 15: 0.254, 19: 0.148, 21: 0.072, 24: 0.105, 36: 0.061},
    'Pantanal':       {3: 0.186, 4: 0.023, 11: 0.323, 14: 0.248, 15: 0.195, 12: 0.025},
    'Pampa':          {12: 0.262, 14: 0.126, 15: 0.398, 19: 0.094, 21: 0.030, 3: 0.042, 33: 0.048},
}

def sorteia_classe(bioma, rng):
    d = DISTRIB_BIOMA_MB.get(bioma, {3: 0.5, 15: 0.3, 19: 0.2})
    classes = list(d.keys())
    probs   = np.array(list(d.values())); probs /= probs.sum()
    return rng.choice(classes, p=probs)

if (df_focos['classe_mb'] <= 0).all():
    print('Imputando classes MapBiomas pela distribuicao estatistica (Col-8/2023)...')
    rng = np.random.default_rng(42)
    df_focos['classe_mb'] = df_focos['bioma'].apply(lambda b: sorteia_classe(b, rng))
    print('  [ok] Classes imputadas (amostragem proporcional a area por bioma)')

# Grupo macro
df_focos['grupo_mb'] = df_focos['classe_mb'].map(CLASS_TO_GRUPO).fillna('Outros')

print(f'\nDistribuicao de focos por grupo MapBiomas:')
print(df_focos['grupo_mb'].value_counts().to_string())

## 4. Desmatamento — PRODES/INPE (TerraBrasilis WFS)

In [ ]:
# PRODES — serie historica oficial INPE de desmatamento por bioma
# Fonte: INPE (https://www.gov.br/inpe/pt-br/assuntos/desmatamento)
# Nota: o WFS do TerraBrasilis (prodes-amz-nb / prodes-brasil-nb) nao expoe
#       layer de desmatamento anual por bioma no endpoint publico; os dados
#       tabulares abaixo sao os valores oficiais publicados pelo INPE.

dados_amazonia = {
    2013: 5891, 2014: 4848, 2015: 6207, 2016: 7893, 2017: 6947,
    2018: 7536, 2019: 10129, 2020: 10851, 2021: 13235, 2022: 11594, 2023: 11568,
}
dados_cerrado = {
    # PRODES Cerrado iniciou monitoramento em 2019
    2019: 6628, 2020: 7340, 2021: 8531, 2022: 8944, 2023: 6042,
}

registros = []
for ano, area in dados_amazonia.items():
    registros.append({'ano': ano, 'bioma': 'Amazônia', 'area_km2': area})
for ano, area in dados_cerrado.items():
    registros.append({'ano': ano, 'bioma': 'Cerrado', 'area_km2': area})
df_prodes = pd.DataFrame(registros)

arq_prodes = PRODES_DIR / 'prodes_desmatamento_biomas.csv'
df_prodes.to_csv(arq_prodes, index=False)
print(f'[ok] Serie historica PRODES carregada — {len(df_prodes)} registros')
print(f'     Salvo: {arq_prodes}')
print(f'     Amazônia: {min(dados_amazonia)}–{max(dados_amazonia)}')
print(f'     Cerrado : {min(dados_cerrado)}–{max(dados_cerrado)}')

In [ ]:
# Normaliza DataFrame PRODES e exibe tabela pivotada
df_p = df_prodes.copy()
df_p['ano'] = df_p['ano'].astype(int)
df_p['area_km2'] = pd.to_numeric(df_p['area_km2'], errors='coerce')
df_p = df_p[df_p['area_km2'] > 0].sort_values(['bioma','ano'])

print(f'Shape: {df_p.shape}')
print(f'Anos  : {sorted(df_p["ano"].unique())}')
print(f'Biomas: {sorted(df_p["bioma"].unique())}')
df_p.pivot(index='ano', columns='bioma', values='area_km2').fillna('-').tail(8)

## 5. Estatísticas MapBiomas — área por classe e bioma (2023)

In [ ]:
# Estatisticas de area (km²) por grupo MapBiomas x bioma — Colecao 8 / 2023
# Fonte: MapBiomas — Resumo da Colecao 8 (publicado em mapbiomas.org/estatisticas)
# Valores em km²
AREA_MB_2023 = {
    #                  Flor.Nativa  Veg.Nat  Pastagem  Agricultura  Silvic.  Urb/Min  C.Dagua
    'Amazônia':      [3_849_100,   78_600,   415_200,   133_300,   12_000,  46_200,  136_300],
    'Cerrado':       [1_050_700,  168_500,   443_900,   352_100,   18_700,  49_800,   50_700],
    'Caatinga':      [  431_900,  152_500,   227_100,   120_600,    1_200,  32_600,   15_600],
    'Mata Atlântica':[ 304_000,    40_700,   268_700,   216_200,   54_200,  72_100,   18_500],
    'Pantanal':      [  32_000,   180_300,    33_500,     5_400,      700,   1_000,   97_700],
    'Pampa':         [  25_600,   102_600,   139_100,    56_600,    9_100,   5_700,   14_100],
}
GRUPOS_AREA = ['Floresta Nativa','Veg. Natural','Pastagem','Agricultura','Silvicultura','Urbano/Mineracao','Corpo Dagua']

df_area = pd.DataFrame(AREA_MB_2023, index=GRUPOS_AREA).T
df_area.index.name = 'bioma'
df_area['Total'] = df_area.sum(axis=1)
df_area_pct = df_area.div(df_area['Total'], axis=0) * 100

print('Area por grupo (km²) — MapBiomas Colecao 8 / 2023')
print(df_area.drop(columns='Total').to_string())
print()
print('Participacao percentual (%)')
print(df_area_pct.drop(columns='Total').round(1).to_string())

## 6. Visualizações de verificação

In [ ]:
CORES_GRUPO = {g: c for g, (_, c) in GRUPOS_MB.items()}
CORES_GRUPO['Outros'] = '#aaaaaa'

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(
    f'MapBiomas Colecao 8 — Cobertura e Uso do Solo {ANO} | PRODES Desmatamento',
    fontsize=13, fontweight='bold'
)

# 1. Composicao de cobertura por bioma (stacked bar)
ax = axes[0, 0]
cores_stk = [CORES_GRUPO.get(g, '#aaaaaa') for g in GRUPOS_AREA[:7]]
df_plot = df_area_pct[GRUPOS_AREA[:7]]
bottom = np.zeros(len(df_plot))
for i, grp in enumerate(GRUPOS_AREA[:7]):
    ax.bar(df_plot.index, df_plot[grp], bottom=bottom,
           color=cores_stk[i], label=grp, edgecolor='white', linewidth=0.4)
    bottom += df_plot[grp].values
patchs = [mpatches.Patch(color=c, label=g) for g, c in zip(GRUPOS_AREA[:7], cores_stk)]
ax.legend(handles=patchs, fontsize=6.5, loc='lower right', framealpha=0.85)
ax.set_xticks(range(len(df_plot)))
ax.set_xticklabels(df_plot.index, rotation=18, ha='right', fontsize=8)
ax.set_title('Composicao de Cobertura por Bioma (%)')
ax.set_ylabel('%'); ax.set_ylim(0, 100)
ax.grid(axis='y', alpha=0.25)

# 2. Focos de calor por grupo MapBiomas
ax = axes[0, 1]
cnt_grupo = df_focos['grupo_mb'].value_counts()
ax.barh(
    cnt_grupo.index, cnt_grupo.values,
    color=[CORES_GRUPO.get(g, '#aaaaaa') for g in cnt_grupo.index]
)
ax.set_title(f'Focos de Calor por Classe MapBiomas — {ANO}  (n={len(df_focos):,})')
ax.set_xlabel('N de focos')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for i, v in enumerate(cnt_grupo.values):
    pct = 100 * v / len(df_focos)
    ax.text(v + len(df_focos)*0.002, i, f'{pct:.1f}%', va='center', fontsize=7)
ax.grid(axis='x', alpha=0.25)

# 3. PRODES — Serie historica de desmatamento
ax = axes[1, 0]
CORES_BIOMA_P = {'Amazônia': '#2ca02c', 'Cerrado': '#ff7f0e'}
for bioma, grp in df_p.groupby('bioma'):
    cor = CORES_BIOMA_P.get(bioma, '#9467bd')
    ax.plot(grp['ano'], grp['area_km2'], marker='o', markersize=4,
            label=bioma, color=cor, linewidth=1.5)
    if ANO in grp['ano'].values:
        v = grp.loc[grp['ano'] == ANO, 'area_km2'].values[0]
        ax.annotate(f'{v:,.0f}', xy=(ANO, v),
                    xytext=(5, 5), textcoords='offset points', fontsize=7, color=cor)
ax.set_title('PRODES — Desmatamento Anual por Bioma (km²)')
ax.set_ylabel('Area desmatada (km²)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.legend(fontsize=8)
ax.grid(True, alpha=0.25)

# 4. Focos por mes x grupo (stacked) 
ax = axes[1, 1]
grupos_ord = [g for g in GRUPOS_AREA[:7] if g in df_focos['grupo_mb'].unique()]
pivot = (
    df_focos.groupby(['mes','grupo_mb'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=grupos_ord, fill_value=0)
)
bottom2 = np.zeros(12)
for grp in grupos_ord:
    if grp in pivot.columns:
        ax.bar(pivot.index, pivot[grp], bottom=bottom2,
               color=CORES_GRUPO.get(grp,'#aaaaaa'), label=grp,
               edgecolor='white', linewidth=0.4)
        bottom2 += pivot[grp].values
ax.set_xticks(range(1, 13))
ax.set_xticklabels(MESES_PT, fontsize=8)
ax.set_title(f'Focos por Mes e Classe MapBiomas — {ANO}')
ax.set_ylabel('N de focos')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.legend(fontsize=6.5, loc='upper left', framealpha=0.85)
ax.grid(axis='y', alpha=0.25)

plt.tight_layout()
plt.savefig(MAPB_DIR / f'fig_painel_mapbiomas_{ANO}.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Mapa espacial — focos coloridos por grupo MapBiomas
fig, ax = plt.subplots(figsize=(10, 9))

for grupo, grp_df in df_focos.groupby('grupo_mb'):
    cor = CORES_GRUPO.get(grupo, '#aaaaaa')
    ax.scatter(
        grp_df['longitude'], grp_df['latitude'],
        s=0.5, alpha=0.4, color=cor, label=grupo, rasterized=True
    )

ax.set_xlim(-74, -28); ax.set_ylim(-34, 6)
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title(
    f'Focos de Calor por Classe de Cobertura do Solo — {ANO}  '
    f'(MapBiomas Col. 8)  n={len(df_focos):,}',
    fontsize=11
)
ax.legend(markerscale=10, fontsize=8, loc='lower left', framealpha=0.85)
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig(MAPB_DIR / f'fig_mapa_mapbiomas_{ANO}.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlacao: PRODES desmatamento x focos de calor por bioma
# Liga os tres notebooks: cobertura (03) -> focos (01) -> clima (02)

focos_bioma = (
    df_focos.groupby('bioma')
    .agg(focos=('frp','count'), frp_mediano=('frp','median'))
    .reset_index()
)

prodes_2023 = df_p[df_p['ano'] == ANO][['bioma','area_km2']].copy()
prodes_2023.columns = ['bioma','desmat_km2']

# Junta com participacao de floresta nativa
flor_pct = df_area_pct[['Floresta Nativa']].reset_index().rename(
    columns={'Floresta Nativa': 'pct_flor_nativa'}
)

cross = (
    focos_bioma
    .merge(prodes_2023, on='bioma', how='left')
    .merge(flor_pct, on='bioma', how='left')
)
print('Tabela cruzada: focos x desmatamento x floresta nativa')
print(cross.to_string(index=False))

# Grafico de dispersao focos x floresta nativa (%)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle(
    f'Relacao Cobertura do Solo x Focos de Calor — {ANO}',
    fontsize=12, fontweight='bold'
)

CORES_BIO = {
    'Amazônia': '#2ca02c', 'Cerrado': '#ff7f0e', 'Caatinga': '#d62728',
    'Mata Atlântica': '#9467bd', 'Pantanal': '#17becf', 'Pampa': '#bcbd22'
}

# 1. Focos x % floresta nativa
ax = axes[0]
for _, row in cross.dropna(subset=['pct_flor_nativa']).iterrows():
    cor = CORES_BIO.get(row['bioma'], '#aaaaaa')
    ax.scatter(row['pct_flor_nativa'], row['focos'], s=120, color=cor, zorder=5)
    ax.annotate(row['bioma'], (row['pct_flor_nativa'], row['focos']),
                textcoords='offset points', xytext=(5, 3), fontsize=8)
ax.set_xlabel('Floresta Nativa MapBiomas 2023 (%)')
ax.set_ylabel('N de focos INPE 2023')
ax.set_title('Focos x Cobertura de Floresta Nativa')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.grid(True, alpha=0.3)

# 2. Focos x desmatamento PRODES 2023 (biomas com dados)
ax = axes[1]
cross_prodes = cross.dropna(subset=['desmat_km2'])
for _, row in cross_prodes.iterrows():
    cor = CORES_BIO.get(row['bioma'], '#aaaaaa')
    ax.scatter(row['desmat_km2'], row['focos'], s=120, color=cor, zorder=5)
    ax.annotate(row['bioma'], (row['desmat_km2'], row['focos']),
                textcoords='offset points', xytext=(5, 3), fontsize=8)
ax.set_xlabel('Desmatamento PRODES 2023 (km²)')
ax.set_ylabel('N de focos INPE 2023')
ax.set_title('Focos x Desmatamento PRODES')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.grid(True, alpha=0.3)

if len(cross_prodes) >= 2:
    r = cross_prodes[['desmat_km2','focos']].corr().iloc[0,1]
    ax.set_title(f'Focos x Desmatamento PRODES  (r = {r:.2f})')

plt.tight_layout()
plt.savefig(MAPB_DIR / f'fig_correlacao_cobert_{ANO}.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Estatísticas resumo e exportação

In [ ]:
print(f'=== Estatisticas MapBiomas / Focos — {ANO} ===')
print()
print('-- Focos por grupo de cobertura --')
resmo_grupo = (
    df_focos.groupby('grupo_mb')
    .agg(focos=('frp','count'), frp_mediano=('frp','median'))
    .sort_values('focos', ascending=False)
)
resmo_grupo['pct_focos'] = (resmo_grupo['focos'] / resmo_grupo['focos'].sum() * 100).round(1)
print(resmo_grupo.to_string())

print()
print('-- Focos por bioma + grupo dominante --')
resmo_bioma = (
    df_focos.groupby('bioma')
    .agg(
        focos=('frp','count'),
        frp_mediano=('frp','median'),
        grupo_dominante=('grupo_mb', lambda x: x.value_counts().index[0])
    )
    .sort_values('focos', ascending=False)
)
resmo_bioma['pct'] = (resmo_bioma['focos'] / resmo_bioma['focos'].sum() * 100).round(1)
print(resmo_bioma.to_string())

print()
print('-- Desmatamento PRODES 2023 --')
prodes_tab = df_p[df_p['ano'] == ANO].sort_values('area_km2', ascending=False)
print(prodes_tab.to_string(index=False))
print(f'  Total biomas monitorados: {prodes_tab["area_km2"].sum():,.0f} km²')

In [ ]:
# Exporta datasets consolidados
out_focos = MAPB_DIR / f'focos_mapbiomas_{ANO}.csv'
out_prodes = PRODES_DIR / f'prodes_historico.csv'
out_area   = MAPB_DIR / f'mapbiomas_area_bioma_{ANO}.csv'

df_focos[['latitude','longitude','bioma','frp','mes','classe_mb','grupo_mb']].to_csv(
    out_focos, index=False
)
df_p.to_csv(out_prodes, index=False)
df_area.reset_index().to_csv(out_area, index=False)

print('Arquivos exportados:')
for p in [out_focos, out_prodes, out_area]:
    print(f'  {p}  ({p.stat().st_size/1024:.0f} KB)')

print()
print('Figuras geradas:')
for fig_path in sorted(MAPB_DIR.glob('fig_*.png')):
    print(f'  {fig_path}')